# Notebook 16 — Agentic RL IV: Training a Real Tool-Using Agent (Qwen2.5-0.5B + Calculator)

### Course roadmap

**Part I — classical RL** (`numpy` only)
1. Foundations & MDPs · 2. Dynamic Programming · 3. Monte Carlo & TD · 4. Function Approximation & DQN · 5. Policy Gradients · 6. Trust Regions & PPO

**Part II — RL for language models**
7. RLHF · 8. RLOO & RLVR · 9. GRPO & DeepSeek-R1 · 10. DPO & production libraries · 11. RLCD · 12 / 12.2 / 12.3. Teaching a small model to reason (GRPO, RLCD, PPO)

**Part III — Agentic RL**
- 13. Agentic RL foundations: from reasoning to acting, and why agents need RL
- 14. The agent loop as an RL environment: multi-turn rollouts, tool calls, observation masking
- 15. Credit assignment across turns: trajectory vs step-level advantages, critics, process rewards
- **16. A real tool-using agent: Qwen2.5-0.5B + calculator, multi-turn GRPO in PyTorch and `trl`** ← *you are here*
- 17. Agentic RL at scale: reward hacking, async/off-policy rollouts, frameworks, open problems

### What this notebook does

Notebook 12 taught `Qwen2.5-0.5B-Instruct` to *reason*. Here we teach the same
small model to **act**: to solve word problems by calling a **calculator
tool**, reading its result, calling it again if needed, and committing to a
checkable answer. That's notebooks 13–15 on a real LLM:

```
user:      A warehouse receives 412 crates of pens, each holding 37. It already had 2210 pens. ...
assistant: <tool_call>{"name": "calculator", "arguments": {"expression": "412 * 37 + 2210"}}</tool_call>
tool:      <tool_response>17454</tool_response>          ← written by the environment (mask 0)
assistant: The warehouse now has <answer>17454</answer> pens.
```

We train it twice, with the same reward, model, LoRA and budget:

| | Part 1: `trl` | Part 2: hand-written PyTorch |
|---|---|---|
| multi-turn rollout | `GRPOTrainer(tools=[calculator])`, `max_tool_calling_iterations` | our own loop: generate → parse `<tool_call>` → run the tool → append `<tool_response>` → generate |
| observation masking | `trl`'s internal `tool_mask` | our `mask` list (1 = agent token, 0 = prompt / tool output) |
| loss | `loss_type="dr_grpo"`, `beta=0.04` | the same objective, written out |

Both are evaluated with **one protocol**: our own greedy multi-turn rollout on 48
held-out problems.

> **Hardware.** Runs on a CUDA GPU (bf16), Apple-Silicon (MPS) or, slowly, CPU.
> A CUDA GPU is recommended. On an Apple M4 (MPS, float32) one training step takes
> about a minute, so `STEPS = 40` per part means well over an hour per part. Lower `STEPS`
> for a quick look, or raise it on a bigger GPU for stronger results.

## 1. Setup

In [ ]:
# !pip install -q "torch>=2.4" "transformers>=5.2" "trl>=1.14" "peft>=0.17" datasets accelerate

In [ ]:
import os, re, json, ast, time, random, operator, warnings, gc
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import transformers, trl, peft
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float32

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(0)

STEPS = 40        # RL steps per part (each step = 4 problems x 8 episodes). Raise on a GPU.

print(f"device        : {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"dtype         : {DTYPE}")
for lib in [torch, transformers, trl, peft]:
    print(f"{lib.__name__:14}: {lib.__version__}")

## 2. The task, the tool and the verifier

**Task.** Word problems with 3- and 4-digit numbers. They're easy to set up,
but too big for a 0.5B model to multiply in its head. Three templates:
multiply-then-subtract, multiply-then-add, and multiply-then-**integer-divide**
("keeping the remainder"). The last one needs care: `/` gives a decimal, so the
agent must use `//` or a second step. Train and test sets come from different
seeds.

**Tool.** `calculator(expression)` evaluates `+ - * / // % ( )` with a small AST
walker: never Python's `eval` on model output. Errors come back as text
(`"Error: ..."`), so the agent can read them and retry (notebook 14).
`trl` and the chat template build the JSON tool schema from the function's
**type hints and Google-style docstring**, so the docstring is part of the
interface.

**Verifier.** Strict, as in notebooks 8 and 12: the integer inside
`<answer>…</answer>` in the agent's **final** message. A "lenient" score (the
last number in the final message) is logged for insight only.

In [ ]:
NAMES = ["Sam", "Mia", "Leo", "Ava", "Noah", "Zoe"]
CONTAINERS = ["boxes", "crates", "bags", "packs"]
THINGS = ["pens", "apples", "screws", "cards"]

def make_problem(rng):
    n, box, th = rng.choice(NAMES), rng.choice(CONTAINERS), rng.choice(THINGS)
    a, b, c, d = rng.randint(120, 980), rng.randint(13, 97), rng.randint(1000, 9000), rng.randint(3, 9)
    kind = rng.randint(0, 2)
    if kind == 0:
        q = f"{n} has {a} {box} with {b} {th} in each. {n} gives away {c} {th}. How many {th} are left?"
        ans = a * b - c
    elif kind == 1:
        q = f"A warehouse receives {a} {box} of {th}, each holding {b}. It already had {c} {th}. How many {th} does it have now?"
        ans = a * b + c
    else:
        q = (f"{n} packs {a} {th} into each of {b} {box}, then splits all of them equally among {d} stores, "
             f"keeping the remainder. How many {th} does each store get?")
        ans = (a * b) // d
    return q, ans

TRAIN = [make_problem(random.Random(i)) for i in range(400)]
TEST = [make_problem(random.Random(10_000 + i)) for i in range(48)]

def calculator(expression: str) -> str:
    '''Evaluate an arithmetic expression and return the result.

    Args:
        expression: An arithmetic expression using numbers and + - * / // % ( ), for example "37 * 412 - 19".

    Returns:
        The numeric result as a string.
    '''
    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
           ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod, ast.USub: operator.neg}
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ops:
            return ops[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ops:
            return ops[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    try:
        v = ev(ast.parse(str(expression).replace(",", ""), mode="eval").body)
        return str(int(v)) if float(v).is_integer() else f"{v:.6g}"
    except Exception as e:
        return f"Error: {e}"

ANSWER_RE = re.compile(r"<answer>\s*(-?[\d,]+)\s*</answer>")
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def parse_answer(text):
    m = ANSWER_RE.search(text or "")
    return int(m.group(1).replace(",", "")) if m else None

def lenient_answer(text):
    nums = re.findall(r"-?\d[\d,]*", text or "")
    return int(nums[-1].replace(",", "")) if nums else None

def parse_tool_calls(text):
    '''Expressions of all calculator calls in a message; None for a malformed call.'''
    calls = []
    for m in TOOL_CALL_RE.finditer(text):
        try:
            d = json.loads(m.group(1))
            calls.append(d["arguments"]["expression"] if d.get("name") == "calculator" else None)
        except Exception:
            calls.append(None)
    return calls

# ---- always test the tool and the verifier before RL optimizes against them ----
assert calculator("412 * 37 + 2210") == "17454"
assert calculator("(173 * 33) // 5") == "1141" and calculator("173 * 33 / 5") == "1141.8"
assert calculator("__import__('os')").startswith("Error")
assert parse_answer("so <answer>17,454</answer>.") == 17454 and parse_answer("17454") is None
assert parse_tool_calls('<tool_call>\n{"name": "calculator", "arguments": {"expression": "2*3"}}\n</tool_call>') == ["2*3"]
assert parse_tool_calls("<tool_call>{oops}</tool_call>") == [None]
print("tool and verifier tests passed")
print("example:", TRAIN[0])

## 3. The chat template with tools, and the observation format

`Qwen2.5-Instruct` was trained for tool calling. Passing `tools=[calculator]` to
`apply_chat_template` injects the JSON schema into the system prompt, and the
model answers with `<tool_call>{json}</tool_call>`. Tool results go back as a
`tool` message, which Qwen renders as a **user turn** wrapped in
`<tool_response>`.

Our rollout must append **exactly** the text the template would produce. If
our hand-written observation differs from the template by even one newline,
the model sees a format at training time that it'll never see in deployment.
So we check it:

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")

SYSTEM_PROMPT = ("You are a careful assistant with access to a calculator tool. Always use the calculator for arithmetic. "
                 "When you have the result, reply with the final answer in the form <answer>NUMBER</answer>.")

def messages_for(question):
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

def build_prompt(question):
    return tokenizer.apply_chat_template(messages_for(question), tools=[calculator], add_generation_prompt=True, tokenize=False)

def observation_text(results):
    '''What the environment appends after an assistant turn that ended with tool calls.'''
    body = "\n".join(f"<tool_response>\n{r}\n</tool_response>" for r in results)
    return f"\n<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\n"

# ---- check: prompt + agent turn + our observation == the chat template's rendering ----
q = TRAIN[0][0]
call = {"type": "function", "function": {"name": "calculator", "arguments": {"expression": "1+1"}}}
templ = tokenizer.apply_chat_template(messages_for(q) + [{"role": "assistant", "content": "", "tool_calls": [call]},
                                                          {"role": "tool", "name": "calculator", "content": "2"}],
                                      tools=[calculator], add_generation_prompt=True, tokenize=False)
ours = build_prompt(q) + '<tool_call>\n{"name": "calculator", "arguments": {"expression": "1+1"}}\n</tool_call><|im_end|>' + observation_text(["2"])
assert templ == ours, "observation format differs from the chat template!"
print("observation format matches the chat template exactly\n")
print(build_prompt(q)[-700:])

## 4. The multi-turn rollout engine

The same algorithm as notebook 14, now at the granularity real systems use:

```
for turn in 1..MAX_TURNS:
    generate (batched, only the still-active episodes) until <|im_end|>      → mask 1
    if the message contains <tool_call>s and turns remain:
        run the tool(s), append observation_text(results)                     → mask 0
    else:
        the episode is finished; its last message is the final answer
```

Each episode keeps **one flat list of token ids and a parallel mask**. The
training step later runs a single forward pass over the whole sequence and uses
the mask to pick out the agent's tokens. For simplicity we re-encode the whole
context at every turn. Production systems keep the KV cache between turns
(vLLM / SGLang with prefix caching).

In [ ]:
MAX_TURNS = 3          # up to 2 tool rounds + 1 final message (trl: max_tool_calling_iterations = 2)
MAX_NEW = 160          # tokens per assistant message

@torch.no_grad()
def rollout(model, questions, answers, greedy=False, temperature=1.0):
    '''Run one episode per question. Returns a list of dicts (ids, mask, prompt_len, final, stats).'''
    model.eval()
    eps = []
    for q, a in zip(questions, answers):
        ids = tokenizer(build_prompt(q), add_special_tokens=False).input_ids
        eps.append(dict(ids=ids, mask=[0] * len(ids), prompt_len=len(ids), answer=a, final="",
                        calls=0, bad_calls=0, truncated=False, turns=0))
    active = list(range(len(eps)))
    gen_kw = dict(do_sample=False) if greedy else dict(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
    for turn in range(MAX_TURNS):
        if not active:
            break
        batch = [eps[i]["ids"] for i in active]
        L = max(map(len, batch))
        inp = torch.tensor([[tokenizer.pad_token_id] * (L - len(x)) + x for x in batch], device=model.device)
        att = (torch.arange(L)[None] >= torch.tensor([L - len(x) for x in batch])[:, None]).long().to(model.device)
        out = model.generate(input_ids=inp, attention_mask=att, max_new_tokens=MAX_NEW, eos_token_id=IM_END,
                             pad_token_id=tokenizer.pad_token_id, **gen_kw)[:, L:].tolist()
        still = []
        for i, g in zip(active, out):
            e = eps[i]
            ended = IM_END in g
            g = g[:g.index(IM_END) + 1] if ended else [t for t in g if t != tokenizer.pad_token_id]
            e["ids"] += g; e["mask"] += [1] * len(g); e["turns"] += 1          # agent tokens
            text = tokenizer.decode(g, skip_special_tokens=True)
            calls = parse_tool_calls(text)
            if calls and ended and turn < MAX_TURNS - 1:                      # execute the tool calls
                results = [calculator(c) if c is not None else "Error: could not parse the tool call" for c in calls]
                e["calls"] += len(calls); e["bad_calls"] += sum(c is None for c in calls)
                obs = tokenizer(observation_text(results), add_special_tokens=False).input_ids
                e["ids"] += obs; e["mask"] += [0] * len(obs)                  # environment tokens
                still.append(i)
            else:                                                             # final message (or cut off)
                e["final"], e["truncated"] = text, (not ended) or bool(calls)
        active = still
    for e in eps:
        e["correct"] = float(parse_answer(e["final"]) == e["answer"])
        e["has_answer_tag"] = float(parse_answer(e["final"]) is not None)
    return eps

def transcript(e):
    return tokenizer.decode(e["ids"][e["prompt_len"]:], skip_special_tokens=False).replace("<|endoftext|>", "")

def evaluate(model, label, problems=TEST, batch_size=16, show=2):
    t0 = time.time()
    eps = []
    for i in range(0, len(problems), batch_size):
        chunk = problems[i:i + batch_size]
        eps += rollout(model, [q for q, _ in chunk], [a for _, a in chunk], greedy=True)
    metrics = {
        "accuracy (strict)": np.mean([e["correct"] for e in eps]),
        "accuracy (lenient)": np.mean([lenient_answer(e["final"]) == e["answer"] for e in eps]),
        "<answer> tag used": np.mean([e["has_answer_tag"] for e in eps]),
        "used the tool": np.mean([e["calls"] > 0 for e in eps]),
        "tool calls / episode": np.mean([e["calls"] for e in eps]),
        "malformed calls / episode": np.mean([e["bad_calls"] for e in eps]),
        "truncated": np.mean([e["truncated"] for e in eps]),
        "agent tokens / episode": np.mean([sum(e["mask"]) for e in eps]),
        "tool-output tokens / episode": np.mean([len(e["mask"]) - e["prompt_len"] - sum(e["mask"]) for e in eps]),
    }
    print(f"===== {label}: {len(problems)} held-out problems in {time.time() - t0:.0f}s =====")
    for k, v in metrics.items():
        print(f"   {k:30}: {v:.3f}" if v < 10 else f"   {k:30}: {v:.1f}")
    for e, (q, a) in list(zip(eps, problems))[:show]:
        print(f"\n   --- {q}\n   --- true answer: {a}\n{transcript(e)}")
    return metrics, eps

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
before_metrics, before_eps = evaluate(base_model, "BEFORE training (base model)")

Read the "before" transcripts. The base model is already a decent tool
*caller* (it was instruction-tuned for it), so this isn't about teaching
the JSON format. The typical failures are **agentic**:

- it answers in prose ("there are 29175 pens left") and never commits to the
  `<answer>` format, so the strict verifier gives 0;
- on the "keeping the remainder" problems it uses `/`, gets a decimal, and
  reports it instead of fixing the expression with `//` or a second call;
- sometimes it stops after the first call or writes a long derivation instead
  of calling the tool.

These are exactly the behaviors RL can fix: they **show up sometimes** in
sampling, and they're **checkable** from the outcome.

## 5. Rewards

Two rewards, summed (as in notebook 12):

- **correctness**: 1.0 if the final `<answer>` is right;
- **format**: 0.2 if the final message contains an `<answer>NUMBER</answer>` tag
  at all. That's partial credit, so early groups aren't all zero
  (notebook 12, Section 3).

No reward for *calling the tool*. Notebook 15 showed how rewarding behavior
that merely correlates with success gets farmed. The tool is used because it
**causes** correct answers.

In [ ]:
CORRECT_W, FORMAT_W = 1.0, 0.2

def episode_reward(e):
    return CORRECT_W * e["correct"] + FORMAT_W * e["has_answer_tag"]

# ---- trl-style reward functions: `completions` is the list of messages the agent produced, incl. tool turns ----
def final_assistant_text(messages):
    for m in reversed(messages):
        if m["role"] == "assistant" and m.get("content"):
            return m["content"]
    return ""

def correctness_reward(completions, answer, **kwargs):
    return [CORRECT_W * float(parse_answer(final_assistant_text(c)) == a) for c, a in zip(completions, answer)]

def format_reward(completions, **kwargs):
    return [FORMAT_W * float(parse_answer(final_assistant_text(c)) is not None) for c in completions]

demo = [{"role": "assistant", "content": "", "tool_calls": [call]}, {"role": "tool", "name": "calculator", "content": "2"},
        {"role": "assistant", "content": "So it is <answer>2</answer>."}]
print(correctness_reward([demo], [2]), format_reward([demo]))

# Part 1 — `trl`'s `GRPOTrainer` with tools

`trl` (≥ 1.x) runs the multi-turn loop for you. Pass `tools=[calculator]`
and it will:

1. render the tool schema into each prompt;
2. generate, parse `<tool_call>` blocks with the model's chat template, run the
   tool, append a `tool` message, and generate again, up to
   `max_tool_calling_iterations` rounds;
3. build a **`tool_mask`** (0 on tool-result tokens, 1 elsewhere) and apply it
   to the loss, the KL and the logged lengths: notebook 14's observation
   masking;
4. hand the reward functions the full list of agent messages, including the tool turns.

Everything else is the same `GRPOConfig` as notebook 12.

In [ ]:
train_dataset = Dataset.from_list([{"prompt": messages_for(q), "answer": a} for q, a in TRAIN])

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM")

grpo_config = GRPOConfig(
    output_dir="/tmp/nb16_trl_agent",
    num_generations=8,                     # G episodes per problem
    per_device_train_batch_size=8,         # episodes per forward/backward micro-batch (memory)
    gradient_accumulation_steps=4,         # 4 x 8 = 32 episodes per optimizer step = 4 problems x 8 episodes
    max_completion_length=512,             # the whole multi-turn completion (agent + tool tokens)
    max_tool_calling_iterations=MAX_TURNS - 1,
    temperature=1.0,
    loss_type="dr_grpo", scale_rewards=False, beta=0.04, epsilon=0.2,
    learning_rate=1e-5, max_steps=STEPS, logging_steps=1,
    report_to="none", save_strategy="no", seed=0,
    bf16=USE_BF16, use_cpu=(DEVICE == "cpu"),
)

base_model.cpu(); del base_model; gc.collect()       # free memory: the "before" model is no longer needed
trl_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
trainer = GRPOTrainer(model=trl_model, reward_funcs=[correctness_reward, format_reward], args=grpo_config,
                      train_dataset=train_dataset, tools=[calculator], peft_config=lora_config)
print(f"{STEPS} steps x 4 problems x 8 episodes = {STEPS * 32} training episodes")

In [ ]:
t0 = time.time()
trainer.train()
trl_minutes = (time.time() - t0) / 60
print(f"trl training finished in {trl_minutes:.1f} min")

log = [h for h in trainer.state.log_history if "reward" in h]
keys = ["rewards/correctness_reward/mean", "rewards/format_reward/mean", "tools/call_frequency",
        "tools/failure_frequency", "completions/mean_length", "kl", "frac_reward_zero_std"]
print(f"\n{'step':>4} | " + " | ".join(f"{k.split('/')[-2] if k.count('/') > 1 else k.split('/')[-1]:>18}" for k in keys))
for i, h in enumerate(log):
    if i % max(1, len(log) // 10) == 0 or i == len(log) - 1:
        print(f"{i:>4} | " + " | ".join(f"{float(h.get(k, float('nan'))):>18.3f}" for k in keys))

In [ ]:
trl_policy = trainer.model.to(DEVICE).eval()
trl_metrics, trl_eps = evaluate(trl_policy, "AFTER trl GRPO (tools)")

### `trl`'s logged columns, read as an agent trainer

| column | meaning | what to look for |
|---|---|---|
| `correctness_reward` / `format_reward` | mean rewards per step | format rises first (it's easier), then correctness |
| `tools/call_frequency` | tool calls per episode | should settle where the task needs it (1–2 here); a steady climb with flat reward is a sign of tool spam |
| `tools/failure_frequency` | share of calls that failed to parse or raised | should fall: the agent learns the call format and valid expressions |
| `completions/mean_length` | model-generated tokens only (`tool_mask = 1`) | the tool output isn't counted, which is the mask at work |
| `frac_reward_zero_std` | groups with no reward variance | high early (all fail) or late (all succeed) means little signal |

Freeing the `trl` model before Part 2 keeps memory low on small machines.

In [ ]:
trl_policy.cpu(); del trainer, trl_model, trl_policy
gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()
if DEVICE == "mps": torch.mps.empty_cache()

# Part 2 — Multi-turn GRPO from scratch in PyTorch

The same algorithm without the library: a `nn.Module` policy with LoRA,
our `rollout`, and a loss computed **only on the agent's tokens**.

**Log-probabilities of an interleaved sequence.** One forward pass over the
whole right-padded sequence (prompt + agent + tool tokens). Position $k-1$
predicts token $k$. We gather $\log\pi_\theta(y_k \mid y_{<k})$ for every $k$
after the shortest prompt, and the **mask** zeroes out everything the agent
didn't write. Prompt logits are skipped with `logits_to_keep`, as in
notebook 12.

**Loss.** Dr. GRPO with the agent mask $m_{i,k}$ and $\beta = 0.04$ k3 KL to the
reference (the base model: LoRA adapters disabled):

$$
\mathcal L = \frac{1}{B \cdot L_{\max}} \sum_{i}\sum_{k} m_{i,k}\Big[-\min\big(\rho_{i,k}\hat A_i,\ \mathrm{clip}(\rho_{i,k}, 1\pm\epsilon)\hat A_i\big) + \beta\,\mathrm{k3}_{i,k}\Big],
\qquad \hat A_i = R_i - \bar R_{\text{group}} .
$$

In [ ]:
class AgentPolicy(nn.Module):
    '''LoRA-adapted LM. forward() returns per-token log-probs over the agent region; ref via disabled adapters.'''
    def __init__(self, model_name, lora_cfg):
        super().__init__()
        self.lm = get_peft_model(AutoModelForCausalLM.from_pretrained(model_name, dtype=DTYPE), lora_cfg)
        for p in self.lm.parameters():
            if p.requires_grad: p.data = p.data.float()

    @property
    def device(self):
        return next(self.lm.parameters()).device

    def generate(self, **kw):                         # so `rollout` can drive it like a HF model
        return self.lm.generate(**kw)

    def eval(self):
        self.lm.eval(); return self

    def forward(self, ids, attn, start):
        '''log pi(ids[:, k] | ids[:, <k]) for k >= start. Shape (B, L - start).'''
        L = ids.shape[1]
        logits = self.lm(input_ids=ids, attention_mask=attn, logits_to_keep=L - start + 1).logits[:, :-1]
        targets = ids[:, start:]
        return torch.stack([row.gather(-1, t[:, None]).squeeze(-1).float() - torch.logsumexp(row.float(), -1)
                            for row, t in zip(logits, targets)])

    @torch.no_grad()
    def ref_forward(self, ids, attn, start):
        with self.lm.disable_adapter():
            return self.forward(ids, attn, start)

def collate(episodes):
    '''Right-pad ids and masks; start = first position that can hold an agent token.'''
    L = max(len(e["ids"]) for e in episodes)
    ids = torch.full((len(episodes), L), tokenizer.pad_token_id)
    attn = torch.zeros(len(episodes), L, dtype=torch.long)
    mask = torch.zeros(len(episodes), L)
    for i, e in enumerate(episodes):
        n = len(e["ids"])
        ids[i, :n] = torch.tensor(e["ids"]); attn[i, :n] = 1; mask[i, :n] = torch.tensor(e["mask"], dtype=torch.float)
    start = min(e["prompt_len"] for e in episodes)
    return ids.to(DEVICE), attn.to(DEVICE), mask[:, start:].to(DEVICE), start

def grpo_loss(logp, old_logp, ref_logp, adv, mask, beta=0.04, eps=0.2, L_max=512):
    ratio = torch.exp(logp - old_logp)
    policy_term = -torch.min(ratio * adv[:, None], torch.clamp(ratio, 1 - eps, 1 + eps) * adv[:, None])
    kl = torch.exp(ref_logp - logp) - (ref_logp - logp) - 1                     # k3, reverse KL (notebook 12, FAQ 6)
    per_token = (policy_term + beta * kl) * mask                                 # agent tokens only
    loss = per_token.sum() / (mask.shape[0] * L_max)                             # Dr. GRPO constant normalizer
    return loss, {"kl": ((kl * mask).sum() / mask.sum()).item()}

MICRO_BATCH = 4
P, G = 4, 8

def train_step(policy, optimizer, questions, answers):
    # 1. rollout: G episodes per question, multi-turn, with tool execution
    eps = rollout(policy, [q for q in questions for _ in range(G)], [a for a in answers for _ in range(G)])
    # 2. rewards and group-relative advantages
    R = torch.tensor([episode_reward(e) for e in eps])
    adv = (R.view(-1, G) - R.view(-1, G).mean(1, keepdim=True)).view(-1).to(DEVICE)
    # 3. one optimizer step, in micro-batches (gradient accumulation)
    policy.lm.train(); optimizer.zero_grad()
    kls = []
    for i in range(0, len(eps), MICRO_BATCH):
        ids, attn, mask, start = collate(eps[i:i + MICRO_BATCH])
        logp = policy(ids, attn, start)
        ref_logp = policy.ref_forward(ids, attn, start)
        loss, st = grpo_loss(logp, logp.detach(), ref_logp, adv[i:i + MICRO_BATCH], mask)
        (loss * len(ids) / len(eps)).backward()
        kls.append(st["kl"])
    torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    return {"reward": R.mean().item(), "correct": np.mean([e["correct"] for e in eps]),
            "format": np.mean([e["has_answer_tag"] for e in eps]), "calls": np.mean([e["calls"] for e in eps]),
            "bad_calls": np.mean([e["bad_calls"] for e in eps]),
            "agent_tokens": np.mean([sum(e["mask"]) for e in eps]),
            "tool_tokens": np.mean([len(e["mask"]) - e["prompt_len"] - sum(e["mask"]) for e in eps]),
            "kl": float(np.mean(kls)), "zero_var": (R.view(-1, G).std(1) == 0).float().mean().item()}

set_seed(0)
policy = AgentPolicy(MODEL_NAME, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear",
                                            task_type="CAUSAL_LM")).to(DEVICE)
optimizer = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=1e-5)
print(f"trainable (LoRA) parameters: {sum(p.numel() for p in policy.parameters() if p.requires_grad) / 1e6:.2f}M")

# ---- one step, slowed down: what does a multi-turn batch look like? ----
demo_eps = rollout(policy, [TRAIN[0][0]] * 2, [TRAIN[0][1]] * 2)
ids, attn, mask, start = collate(demo_eps)
print(f"batch ids {tuple(ids.shape)}, agent-region mask {tuple(mask.shape)}, start = {start}")
for e in demo_eps:
    print(f"  episode: prompt {e['prompt_len']} tok | agent {sum(e['mask'])} tok | tool output "
          f"{len(e['mask']) - e['prompt_len'] - sum(e['mask'])} tok | reward {episode_reward(e):.1f}")
with torch.no_grad():
    lp, rlp = policy(ids, attn, start), policy.ref_forward(ids, attn, start)
print("policy == reference before any update?", torch.allclose(lp * mask, rlp * mask, atol=1e-3))

In [ ]:
history, rng = [], random.Random(1)
t0 = time.time()
print(f"{'step':>4} | {'time':>6} | {'reward':>6} | {'correct':>7} | {'format':>6} | {'calls':>5} | {'bad':>5} | "
      f"{'agent tok':>9} | {'tool tok':>8} | {'KL':>7} | {'zero-var':>8}")
for step in range(STEPS):
    batch = rng.sample(TRAIN, P)
    h = train_step(policy, optimizer, [q for q, _ in batch], [a for _, a in batch])
    history.append(h)
    print(f"{step:>4} | {time.time() - t0:5.0f}s | {h['reward']:6.3f} | {h['correct']:7.3f} | {h['format']:6.3f} | "
          f"{h['calls']:5.2f} | {h['bad_calls']:5.2f} | {h['agent_tokens']:9.1f} | {h['tool_tokens']:8.1f} | "
          f"{h['kl']:7.4f} | {h['zero_var']:8.2f}")
manual_minutes = (time.time() - t0) / 60
print(f"\nmanual multi-turn GRPO finished in {manual_minutes:.1f} min")

In [ ]:
manual_metrics, manual_eps = evaluate(policy, "AFTER PyTorch multi-turn GRPO")

## 6. Before vs. after

In [ ]:
rows = {"base model (before)": before_metrics, "trl GRPOTrainer + tools": trl_metrics,
        "PyTorch multi-turn GRPO": manual_metrics}
cols = list(before_metrics.keys())
print(f"{'':28} | " + " | ".join(f"{c[:16]:>16}" for c in cols))
for name, m in rows.items():
    print(f"{name:28} | " + " | ".join(f"{m[c]:16.3f}" for c in cols))

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
names, colors = list(rows), ["gray", "C0", "C1"]
for ax, c in zip(axes, ["accuracy (strict)", "<answer> tag used", "tool calls / episode", "agent tokens / episode"]):
    ax.bar(range(3), [rows[n][c] for n in names], color=colors)
    ax.set_xticks(range(3)); ax.set_xticklabels(["before", "trl", "PyTorch"]); ax.set_title(c)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
sm = lambda v, w=5: np.convolve(v, np.ones(w) / w, "valid")
axes[0].plot(sm([h["correct"] for h in history]), label="PyTorch: correct")
axes[0].plot(sm([h["format"] for h in history]), label="PyTorch: <answer> tag")
axes[0].plot(sm([float(h["rewards/correctness_reward/mean"]) for h in log]), "--", label="trl: correct")
axes[0].set_title("training rollouts"); axes[0].legend(fontsize=8)
axes[1].plot(sm([h["calls"] for h in history]), label="PyTorch")
axes[1].plot(sm([float(h["tools/call_frequency"]) for h in log]), "--", label="trl")
axes[1].set_title("tool calls per episode"); axes[1].legend(fontsize=8)
axes[2].plot(sm([h["kl"] for h in history]), label="PyTorch")
axes[2].plot(sm([float(h["kl"]) for h in log]), "--", label="trl")
axes[2].set_title("KL to the base model (per agent token)"); axes[2].legend(fontsize=8)
for ax in axes: ax.set_xlabel("step")
plt.tight_layout(); plt.show()

In [ ]:
# ---- which problems changed? ----
before_ok = np.array([e["correct"] for e in before_eps]); after_ok = np.array([e["correct"] for e in manual_eps])
print(f"PyTorch agent: fixed {int(((before_ok == 0) & (after_ok == 1)).sum())}, broke {int(((before_ok == 1) & (after_ok == 0)).sum())} "
      f"of {len(TEST)} problems")
kinds = ["subtract", "add", "divide (remainder)"]
kind_of = lambda q: 2 if "remainder" in q else (1 if "already had" in q else 0)
for k in range(3):
    idx = [i for i, (q, _) in enumerate(TEST) if kind_of(q) == k]
    print(f"   {kinds[k]:20}: before {before_ok[idx].mean():.2f} -> after {after_ok[idx].mean():.2f}  ({len(idx)} problems)")

# ---- the same problem, before and after ----
j = next((i for i in range(len(TEST)) if after_ok[i] and not before_ok[i]), 0)
print(f"\n=== {TEST[j][0]}  (answer {TEST[j][1]})\n--- BEFORE ---\n{transcript(before_eps[j])}\n--- AFTER ---\n{transcript(manual_eps[j])}")

## 7. Beyond `tools=`: stateful environments with `environment_factory`

A stateless function is enough for a calculator. Most agent tasks have
**state**: a file system, a shopping cart, a game board, a database the agent
modifies. `trl`'s (experimental) `environment_factory` takes a class:

- **`reset(**row)`** is called with the dataset row before each rollout (it can
  return extra text for the prompt);
- every **public method** becomes a tool (same docstring rules);
- an optional **`get_reward()`** lets the environment score the episode from its
  own internal state. That's the natural place for final-state checks
  ("was the order placed?", "do the tests pass?").

Here's the same task with a stateful environment that also *records* how the
agent used it. A 2-step run just checks that it's wired correctly:

In [ ]:
class CalculatorEnv:
    '''One instance per rollout. Public methods = tools; get_reward() = environment-owned reward.'''
    def reset(self, answer, **row):
        self.answer, self.history = answer, []

    def calculator(self, expression: str) -> str:
        '''Evaluate an arithmetic expression and return the result.

        Args:
            expression: An arithmetic expression using numbers and + - * / // % ( ), for example "37 * 412 - 19".

        Returns:
            The numeric result as a string.
        '''
        result = calculator(expression)
        self.history.append((expression, result))
        return result

    def get_reward(self):
        # environment-side check: did the agent actually compute the right number with the tool?
        return 0.1 * any(r == str(self.answer) for _, r in self.history)

env_config = GRPOConfig(output_dir="/tmp/nb16_env", num_generations=8, per_device_train_batch_size=8,
                        max_completion_length=512, temperature=1.0, loss_type="dr_grpo", scale_rewards=False,
                        beta=0.04, learning_rate=1e-5, max_steps=2, logging_steps=1, report_to="none",
                        save_strategy="no", seed=0, bf16=USE_BF16, use_cpu=(DEVICE == "cpu"))
env_trainer = GRPOTrainer(model=AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE),
                          reward_funcs=[correctness_reward, format_reward], args=env_config,
                          train_dataset=train_dataset, environment_factory=CalculatorEnv, peft_config=lora_config)
env_trainer.train()
print({k: v for k, v in env_trainer.state.log_history[-2].items() if "reward" in k or "tools" in k})
del env_trainer; gc.collect()

For anything that doesn't fit "tools as Python methods" (a remote sandbox,
a browser, a game server), `GRPOTrainer` also accepts a custom **`rollout_func`**
that returns `prompt_ids`, `completion_ids`, `logprobs` and an **`env_mask`**,
which is exactly what our `rollout` + `collate` produce. And dedicated
frameworks (notebook 17) scale the same loop to thousands of parallel
sandboxes.

## 8. Key takeaways & FAQ

**The one-paragraph summary.** A real tool-using agent trains with the same
GRPO as notebook 12. What changes is the rollout. It alternates generation and
tool execution, appends tool results in **exactly the chat template's format**,
and keeps a mask so that only the agent's tokens carry gradient and KL. The base
model could already call the tool. RL turned that into reliable agentic
behavior: call it with the right expression, read the result, finish with a
checkable answer. That's the "RL sharpens what the model already samples"
lesson again (notebooks 8, 12, 13). `trl`'s `tools=` and our hand-written loop
implement the same objective.

**Q1. How do I know the observation mask works?** Two checks in this notebook:
`completions/mean_length` in `trl` counts only model tokens, and our log
shows `agent tok` and `tool tok` separately. A third check worth doing
yourself: compute the loss with a mask of all ones and confirm it changes. If
it doesn't, the mask isn't being applied.

**Q2. Why not reward tool use directly?** Because the agent will optimize the
proxy (notebooks 14–15). Reward the **outcome** and let tool use emerge
because it causes correct answers. If an episode count of tool calls
matters (cost), use a small penalty, ideally only among correct episodes.

**Q3. What happens to the KV cache across turns?** Our loop re-encodes the full
context each turn: simple, but slower. Production rollouts (vLLM / SGLang)
reuse the prefix cache, and asynchronous systems overlap tool execution with
generation for other episodes (notebook 17).

**Q4. Why can the strict and lenient accuracy disagree?** Lenient takes the last
number in the final message, so it credits prose answers the strict verifier
rejects. Before training, the gap shows how much was lost to **format**. After
training, a small gap means the agent commits to the format.

**Q5. Could the agent "hack" this reward?** The obvious attempts fail: the
`<answer>` must contain the exact integer, and the tool can't be talked into
anything but arithmetic. Watch for subtler issues: an agent that calls the
tool with the *answer* it guessed (`calculator("17454")`) gets the same reward
as one that computed it, and that's fine here. In environments where the agent
can **modify what's checked** (tests, files, databases), it isn't fine.
That's the topic of notebook 17.

**Exercises.**

1. Remove the format reward. How many steps does it take before groups stop
   being all-zero?
2. Set `MAX_TURNS = 2` (one tool round). Which problem type suffers, and why?
3. Make the tool flaky (return `"Error: timeout"` 20% of the time). Does the
   agent learn to retry (notebook 13)? Check `tools/failure_frequency`.
4. Port `rollout` to `trl`'s `rollout_func` interface (return `env_mask`), and
   check that it reproduces Part 2's training curve.